# Entity Extraction Pipeline

This notebook demonstrates the NER (Named Entity Recognition) pipeline used by News2World to extract people, places, organizations, and events from raw news text.

In [1]:
import spacy
from collections import Counter

nlp = spacy.load('en_core_web_trf')

def extract_entities(text: str) -> list[dict]:
    """Extract named entities from text using spaCy transformer model."""
    doc = nlp(text)
    entities = []
    for ent in doc.ents:
        if ent.label_ in ('PERSON', 'GPE', 'ORG', 'EVENT', 'LOC'):
            entities.append({
                'text': ent.text,
                'label': ent.label_,
                'start': ent.start_char,
                'end': ent.end_char,
                'confidence': round(ent._.confidence, 3) if hasattr(ent._, 'confidence') else None
            })
    return entities

In [2]:
sample_text = """
In January 2025, a landmark ceasefire agreement was brokered between Israel and Hamas,
mediated by Qatar, Egypt, and the United States. The deal outlined a phased withdrawal
of Israeli forces from Gaza.
"""

entities = extract_entities(sample_text)
print(f'Extracted {len(entities)} entities:')
for e in entities:
    print(f"  {e['text']} ({e['label']})")

Extracted 6 entities:
  Israel (GPE)
  Hamas (ORG)
  Qatar (GPE)
  Egypt (GPE)
  United States (GPE)
  Gaza (GPE)


## Entity Deduplication & Coreference Resolution

We merge duplicate mentions and resolve coreferences before building the knowledge graph.

In [3]:
def deduplicate_entities(entities: list[dict]) -> list[dict]:
    seen = {}
    for e in entities:
        key = e['text'].lower().strip()
        if key not in seen:
            seen[key] = e
    return list(seen.values())

unique = deduplicate_entities(entities)
print(f'Unique entities after dedup: {len(unique)}')

Unique entities after dedup: 6
